# Blind low-level feature extraction — all clips

Reads every `_pose.npy` + `_pose_meta.json` in `roi_pose_dataset/`
(AQA-7, competition, xg2025men once merged).

**What this extracts (complementary to `build_simple_motion_features.py`):**
- Shoulder x-order swap count — proven total-rotation proxy (ρ ≈ +0.43 vs manual turns)
- Cumulative spin — shoulder axis rotation via `np.unwrap`, cut-aware
- **Cork detection** — body inversion fraction (shoulders below hips); ρ ≈ +0.25 from annotations
- Grab detection — wrist-to-ankle proximity, normalised by body height
- Body amplitude — vertical centroid range (proxy for jump height)
- Landing stability — centroid jitter in the last 10% of the clip
- Per-clip quality flags (coverage, duplicate-frame fraction, cut count)

**Design constraints from PROJECT_STATUS.md:**
- Never differentiate across a camera cut (`cut_frames` from `_pose_meta.json`)
- Use p95/max not mean for speed features (oly_beijing clips have 16-30% duplicate frames)
- Whole-clip features only — the automatic air window has median IoU 0.32 vs manual
- Output is a supplement to `pose_features.csv`, not a replacement

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

# ── paths ──────────────────────────────────────────────────────────────────
POSE_DIR  = Path("roi_pose_dataset")
SCORES_CSV = Path("scores/all_clips_scores.csv")
OUT_CSV   = Path("pose_features_blind.csv")

CONF = 0.3   # keypoint confidence threshold

# ── COCO-17 keypoint indices ───────────────────────────────────────────────
NOSE                    = 0
L_EYE,  R_EYE           = 1, 2
L_EAR,  R_EAR           = 3, 4
L_SH,   R_SH            = 5, 6
L_ELB,  R_ELB           = 7, 8
L_WR,   R_WR            = 9, 10
L_HIP,  R_HIP           = 11, 12
L_KNE,  R_KNE           = 13, 14
L_ANK,  R_ANK           = 15, 16

# ── labels — single source ─────────────────────────────────────────────────
sc = pd.read_csv(SCORES_CSV, dtype={"clip_id": str})
sc = sc.rename(columns={"clip_id": "filename", "official_score_100": "target"})
# AQA pose files are named '001','002'… scores CSV uses 'aqa_001','aqa_002'
sc["filename"] = sc["filename"].str.replace("^aqa_", "", regex=True)
labels = sc.copy()

print(f"Labels loaded : {len(labels)} clips  "
      f"(scored: {labels['target'].notna().sum()})")
print(f"  AQA        : {(labels['dataset']=='aqa7').sum()}")
print(f"  competition: {(labels['dataset']=='competition').sum()}")
print(f"Pose dir      : {POSE_DIR.resolve()}")
pose_files = sorted(POSE_DIR.glob("*_pose.npy"))
print(f"Pose files    : {len(pose_files)} found")

## Geometry helpers

In [ ]:
def visible(frame, *indices):
    """True if all keypoints in indices are above CONF."""
    return all(frame[i, 2] >= CONF for i in indices)


def body_height_px(frame):
    """Shoulder-midpoint to ankle-midpoint distance. NaN if any are missing."""
    if not visible(frame, L_SH, R_SH, L_ANK, R_ANK):
        return np.nan
    sh  = (frame[L_SH, :2]  + frame[R_SH, :2])  / 2.0
    ank = (frame[L_ANK, :2] + frame[R_ANK, :2]) / 2.0
    return float(np.linalg.norm(sh - ank))


def body_centroid(frame, min_kpts=4):
    """Confidence-weighted centroid of all visible keypoints."""
    mask = frame[:, 2] >= CONF
    if mask.sum() < min_kpts:
        return np.array([np.nan, np.nan])
    w = frame[mask, 2]
    return (frame[mask, :2] * w[:, None]).sum(0) / w.sum()


def cut_safe_diff(values, cut_set, axis=0):
    """
    Frame-to-frame differences that never cross a camera cut.
    values : array [T, ...] or [T]
    cut_set: set of cut frame indices (from pose_meta['cut_frames'])
    Returns array [T-1, ...] with NaN wherever the gap spans a cut.
    """
    if values.ndim == 1:
        d = np.diff(values).astype(float)
    else:
        d = np.diff(values, axis=0).astype(float)
    for t in range(len(d)):
        if (t + 1) in cut_set:               # frame t -> t+1 crosses a cut
            if d.ndim == 1:
                d[t] = np.nan
            else:
                d[t] = np.nan
    return d


print("Geometry helpers defined.")


## Shoulder x-order swap (total-rotation proxy)

A sign change in `x_left_shoulder - x_right_shoulder` signals a 180-degree
body rotation (the skier passed through facing the camera or away from it).
Two swaps = one full rotation.  Proven ρ ≈ +0.43 vs manual total turns.

Hysteresis (threshold = 10% of frame width) suppresses noise from keypoint
jitter without missing real swaps.  Hip swap is computed as a cross-check.


In [ ]:
def extract_swap_features(pose_seq, meta):
    """
    Hysteretic sign-change count on shoulder and hip x-order.
    Returns swap_turns (shoulder), swap_hip_turns, swap_hip_agree, swap_rate_hz.
    """
    n       = len(pose_seq)
    fps     = float(meta.get("fps") or 59.94)
    width   = int(meta.get("width") or 1280)
    hyst    = width * 0.10              # 10% of frame width

    sh_sign = 0    # 0 = uninitialised
    hip_sign = 0
    sh_swaps  = 0
    hip_swaps = 0
    sh_hip_agree = 0
    sh_hip_checks = 0

    for frame in pose_seq:
        # shoulder x-order
        if visible(frame, L_SH, R_SH):
            diff_sh = float(frame[L_SH, 0] - frame[R_SH, 0])
            if sh_sign == 0:
                sh_sign = 1 if diff_sh >= 0 else -1
            elif sh_sign == 1 and diff_sh < -hyst:
                sh_swaps += 1; sh_sign = -1
            elif sh_sign == -1 and diff_sh > hyst:
                sh_swaps += 1; sh_sign = 1

        # hip x-order
        if visible(frame, L_HIP, R_HIP):
            diff_hip = float(frame[L_HIP, 0] - frame[R_HIP, 0])
            if hip_sign == 0:
                hip_sign = 1 if diff_hip >= 0 else -1
            elif hip_sign == 1 and diff_hip < -hyst:
                hip_swaps += 1; hip_sign = -1
            elif hip_sign == -1 and diff_hip > hyst:
                hip_swaps += 1; hip_sign = 1

        # agreement check (both visible)
        if visible(frame, L_SH, R_SH, L_HIP, R_HIP):
            sh_hip_checks += 1
            sh_ord  = 1 if frame[L_SH,  0] >= frame[R_SH,  0] else -1
            hip_ord = 1 if frame[L_HIP, 0] >= frame[R_HIP, 0] else -1
            sh_hip_agree += int(sh_ord == hip_ord)

    agree_frac = (sh_hip_agree / sh_hip_checks
                  if sh_hip_checks > 0 else np.nan)
    clip_dur_s = n / fps
    return dict(
        swap_turns      = float(sh_swaps  / 2.0),   # 2 swaps = 1 full rotation
        swap_hip_turns  = float(hip_swaps / 2.0),
        swap_hip_agree  = float(agree_frac),
        swap_rate_hz    = float(sh_swaps  / 2.0 / max(clip_dur_s, 1e-6)),
    )


print("Swap extractor defined.")


## Spin: cumulative shoulder-axis rotation (cut-aware)

Tracks the angle of the left→right shoulder vector, unwraps it to a
continuous signal, and reports total rotation and speed.

Camera cuts are respected: frame-to-frame deltas that cross a cut boundary
are excluded so a jump-cut cannot inject a fake rotation.


In [ ]:
def extract_spin_features(pose_seq, meta):
    """
    Cumulative shoulder (or hip fallback) axis rotation, cut-aware.

    Total rotation is computed as the sum of shortest-path frame-to-frame
    angle differences, zeroing out steps that cross a camera cut.  This is
    correct for multi-shot competition clips; the previous np.unwrap approach
    accumulated the camera-angle jump at each cut as fake body rotation.

    Uses p95 for speed to be robust to oly_beijing duplicate frames.
    """
    fps     = float(meta.get("fps") or 59.94)
    cut_set = set(int(c) for c in (meta.get("cut_frames") or []))

    angles = []
    t_idx  = []
    for t, frame in enumerate(pose_seq):
        if visible(frame, L_SH, R_SH):
            v = frame[R_SH, :2] - frame[L_SH, :2]
        elif visible(frame, L_HIP, R_HIP):
            v = frame[R_HIP, :2] - frame[L_HIP, :2]
        else:
            continue
        angles.append(np.degrees(np.arctan2(v[1], v[0])))
        t_idx.append(t)

    nan_result = dict(
        spin_total_deg=np.nan, spin_total_turns=np.nan,
        spin_net_deg=np.nan,   spin_direction=np.nan,
        spin_speed_p95_dps=np.nan, spin_speed_mean_dps=np.nan,
        spin_valid_frames=0)

    if len(angles) < 2:
        return nan_result

    angle_arr = np.array(angles)

    # Shortest-path frame differences — wrap to (-180, +180] to handle ±180° boundary
    raw_diffs = np.diff(angle_arr)
    raw_diffs = (raw_diffs + 180.0) % 360.0 - 180.0

    # Zero out steps that cross a camera cut (camera-angle change ≠ body rotation)
    for i in range(len(t_idx) - 1):
        if t_idx[i + 1] in cut_set:
            raw_diffs[i] = 0.0

    total_rot = float(raw_diffs.sum())
    net_rot   = total_rot                      # signed net rotation
    abs_rot   = float(np.abs(raw_diffs).sum()) # total absolute rotation

    # cut-aware instantaneous speed
    speeds = []
    for i in range(len(t_idx) - 1):
        t0, t1 = t_idx[i], t_idx[i + 1]
        if t1 in cut_set:
            continue
        dt = (t1 - t0) / fps
        if dt > 0:
            speeds.append(abs(raw_diffs[i]) / dt)

    speeds = np.array(speeds) if speeds else np.array([np.nan])
    return dict(
        spin_total_deg      = abs_rot,
        spin_total_turns    = abs_rot / 360.0,
        spin_net_deg        = net_rot,
        spin_direction      = float(np.sign(net_rot)),
        spin_speed_p95_dps  = float(np.nanpercentile(speeds, 95)),
        spin_speed_mean_dps = float(np.nanmean(speeds)),
        spin_valid_frames   = len(angles),
    )


print("Spin extractor defined  (cut-safe total rotation).")


## Cork axis detection

Annotation data shows `is_cork` has ρ = +0.25 (p = 0.01) with score — the second
strongest novel signal after spin count.

**How cork differs from a straight spin in pose data:**
A cork (off-axis rotation) means the skier passes through *inverted* — shoulders drop
below the hips at some point during the rotation.  In image coordinates (Y increases
downward) this means `shoulder_midpoint_Y > hip_midpoint_Y`.

Two complementary signals:
- `inverted_frac`: fraction of frames where the skier is inverted.
  Near-zero for straight spins; > 3% flags a cork.
- `sh_mid_y_std_norm`: std of shoulder midpoint Y / body height — a continuous
  tilt proxy (also rises with jump amplitude, so treat as supplementary).

In [ ]:
CORK_INVERT_THRESH = 0.03   # 3% of frames inverted → call it a cork


def extract_cork_features(pose_seq, meta):
    """
    Cork axis detection via body inversion.

    In a straight spin the skier stays upright: shoulders always above hips
    (smaller image Y).  In a cork the skier passes through inverted, so
    shoulder_midpoint_Y > hip_midpoint_Y for some frames.

    sh_mid_y_std_norm is a continuous tilt proxy but conflates jump amplitude
    with tilt — use inverted_frac as the primary cork signal.
    """
    heights  = np.array([body_height_px(f) for f in pose_seq])
    median_h = float(np.nanmedian(heights))

    nan_result = dict(inverted_frac=np.nan, cork_detected=0,
                      sh_mid_y_std_norm=np.nan)
    if np.isnan(median_h) or median_h < 1:
        return nan_result

    inverted_count  = 0
    hip_check_count = 0
    sh_mid_ys       = []

    for frame in pose_seq:
        if visible(frame, L_SH, R_SH):
            sh_mid_ys.append(float((frame[L_SH, 1] + frame[R_SH, 1]) / 2.0))

        if visible(frame, L_SH, R_SH, L_HIP, R_HIP):
            sh_y  = (frame[L_SH, 1] + frame[R_SH, 1]) / 2.0
            hip_y = (frame[L_HIP, 1] + frame[R_HIP, 1]) / 2.0
            hip_check_count += 1
            if sh_y > hip_y:           # image Y: larger = lower in frame
                inverted_count += 1

    inverted_frac = (float(inverted_count / hip_check_count)
                     if hip_check_count > 0 else np.nan)
    cork_detected = (int(inverted_frac >= CORK_INVERT_THRESH)
                     if not np.isnan(inverted_frac) else 0)
    sh_mid_y_std_norm = (float(np.std(sh_mid_ys) / median_h)
                         if len(sh_mid_ys) >= 4 else np.nan)

    return dict(
        inverted_frac     = inverted_frac,
        cork_detected     = cork_detected,
        sh_mid_y_std_norm = sh_mid_y_std_norm,
    )


print("Cork extractor defined.")


## Grab detection

Wrist-to-ankle proximity normalised by body height.
A grab (skier holds the ski mid-air) is detected when the minimum
wrist-ankle distance drops below 35% of body height.


In [ ]:
GRAB_THRESH = 0.35   # wrist-ankle / body_height threshold


def extract_grab_features(pose_seq, meta):
    fps     = float(meta.get("fps") or 59.94)
    n       = len(pose_seq)
    heights = np.array([body_height_px(f) for f in pose_seq])
    median_h = float(np.nanmedian(heights))

    nan_result = dict(
        min_grab_dist_norm=np.nan, grab_detected=0,
        grab_duration_s=np.nan,   grab_duration_frac=np.nan,
        grab_depth_mean_norm=np.nan)

    if np.isnan(median_h) or median_h < 1:
        return nan_result

    norm_dists = np.full(n, np.nan)
    for t, frame in enumerate(pose_seq):
        candidates = []
        for w in [L_WR, R_WR]:
            for a in [L_ANK, R_ANK]:
                if visible(frame, w, a):
                    d = np.linalg.norm(frame[w, :2] - frame[a, :2])
                    candidates.append(d)
        if candidates:
            norm_dists[t] = min(candidates) / median_h

    min_dist   = float(np.nanmin(norm_dists))
    grab_mask  = norm_dists <= GRAB_THRESH
    grab_frames = int(grab_mask.sum())

    return dict(
        min_grab_dist_norm  = min_dist,
        grab_detected       = int(min_dist <= GRAB_THRESH),
        grab_duration_s     = float(grab_frames / fps),
        grab_duration_frac  = float(grab_frames / n),
        grab_depth_mean_norm= float(np.nanmean(norm_dists[grab_mask]))
                              if grab_frames > 0 else np.nan,
    )


print("Grab extractor defined.")


## Amplitude and landing stability

In [ ]:
def extract_amplitude_features(pose_seq, meta):
    """Vertical centroid range, normalised by body height (proxy for jump height)."""
    heights  = np.array([body_height_px(f) for f in pose_seq])
    median_h = float(np.nanmedian(heights))
    centroids = np.array([body_centroid(f) for f in pose_seq])
    y = centroids[:, 1]       # image-y: smaller = higher in frame
    valid = ~np.isnan(y)

    if valid.sum() < 2 or np.isnan(median_h) or median_h < 1:
        return dict(amplitude_px=np.nan, amplitude_norm=np.nan,
                    peak_height_frame=np.nan)

    y_v   = y[valid]
    rng   = float(y_v.max() - y_v.min())
    peak  = int(np.where(valid)[0][np.argmin(y_v)])
    return dict(
        amplitude_px      = rng,
        amplitude_norm    = rng / median_h,
        peak_height_frame = float(peak),
    )


def extract_landing_stability(pose_seq, meta, tail_frac=0.10):
    """
    Mean frame-to-frame centroid displacement in the last tail_frac of the clip.
    High jitter = rough / unstable landing.  Cut-aware: cuts in the tail are skipped.
    """
    fps     = float(meta.get("fps") or 59.94)
    cut_set = set(int(c) for c in (meta.get("cut_frames") or []))
    n       = len(pose_seq)
    tail_n  = max(2, int(n * tail_frac))
    tail    = pose_seq[-tail_n:]
    tail_offset = n - tail_n

    heights  = np.array([body_height_px(f) for f in pose_seq])
    median_h = float(np.nanmedian(heights))
    centroids = np.array([body_centroid(f) for f in tail])

    diffs = []
    for i in range(len(tail) - 1):
        t_abs = tail_offset + i + 1
        if t_abs in cut_set:
            continue
        c0, c1 = centroids[i], centroids[i + 1]
        if not (np.isnan(c0).any() or np.isnan(c1).any()):
            diffs.append(np.linalg.norm(c1 - c0))

    if not diffs or np.isnan(median_h) or median_h < 1:
        return dict(landing_jitter_px=np.nan, landing_jitter_norm=np.nan,
                    sudden_stop=np.nan)

    jitter = float(np.mean(diffs))

    # Sudden-stop score: ratio of tail displacement to whole-clip displacement.
    # A fall produces near-zero tail displacement (skier on ground / camera cuts away).
    # Normal clip: ratio closer to 1.
    full_centroids = np.array([body_centroid(f) for f in pose_seq])
    full_diffs = []
    for i in range(len(pose_seq) - 1):
        c0, c1 = full_centroids[i], full_centroids[i + 1]
        if not (np.isnan(c0).any() or np.isnan(c1).any()):
            full_diffs.append(np.linalg.norm(c1 - c0))
    full_mean = float(np.mean(full_diffs)) if full_diffs else np.nan

    sudden_stop = (jitter / full_mean if (not np.isnan(full_mean) and full_mean > 0)
                   else np.nan)

    return dict(
        landing_jitter_px   = jitter,
        landing_jitter_norm = jitter / median_h,
        sudden_stop         = sudden_stop,   # low = movement stops → fall proxy
    )


print("Amplitude and landing extractors defined.")


## Batch processing — all clips

In [ ]:
records = []

for pose_path in sorted(POSE_DIR.glob("*_pose.npy")):
    stem      = pose_path.stem.replace("_pose", "")
    meta_path = POSE_DIR / f"{stem}_pose_meta.json"

    pose_seq = np.load(pose_path)              # [T, 17, 3]
    n        = len(pose_seq)

    meta = {}
    if meta_path.is_file():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))

    fps           = float(meta.get("fps") or 59.94)
    pose_coverage = float(meta.get("pose_coverage") or np.nan)
    n_shots       = int(meta.get("n_camera_shots") or 0)
    cut_frames    = list(meta.get("cut_frames") or [])

    # rough duplicate-frame fraction (oly_beijing clips have 16-30%)
    # a duplicate frame has near-zero centroid displacement
    centroids = np.array([body_centroid(f) for f in pose_seq])
    disps = np.linalg.norm(np.diff(centroids, axis=0), axis=1)
    valid_disps = disps[~np.isnan(disps)]
    dup_frac = float(np.mean(valid_disps < 0.5)) if len(valid_disps) > 0 else np.nan

    # match to labels (may be unscored)
    row = labels[labels["filename"] == stem] if len(labels) else pd.DataFrame()
    target   = float(row["target"].values[0]) if len(row) and "target" in row else np.nan
    fall     = int(row["fall"].values[0])     if len(row) and "fall"   in row else np.nan
    dataset  = str(row["dataset"].values[0])  if len(row) and "dataset" in row else "unknown"

    rec = {
        "filename"        : stem,
        "dataset"         : dataset,
        "target"          : target,
        "fall"            : fall,
        "n_frames"        : n,
        "fps"             : fps,
        "clip_duration_s" : n / fps,
        "pose_coverage"   : pose_coverage,
        "n_camera_shots"  : n_shots,
        "n_cut_frames"    : len(cut_frames),
        "dup_frame_frac"  : dup_frac,
    }

    rec.update(extract_swap_features(pose_seq, meta))
    rec.update(extract_spin_features(pose_seq, meta))
    rec.update(extract_cork_features(pose_seq, meta))
    rec.update(extract_grab_features(pose_seq, meta))
    rec.update(extract_amplitude_features(pose_seq, meta))
    rec.update(extract_landing_stability(pose_seq, meta))

    records.append(rec)
    print(f"  {stem[:45]:45s}  swaps={rec['swap_turns']:.1f}  "
          f"cork={rec['cork_detected']}({rec['inverted_frac']:.2f})  "
          f"grab={rec['grab_detected']}  "
          f"amp={rec['amplitude_norm']:.2f}  "
          f"cov={pose_coverage:.2f}  "
          f"dup={dup_frac:.2f}", flush=True)

df = pd.DataFrame(records)
df.to_csv(OUT_CSV, index=False)
print(f"\nSaved {len(df)} rows → {OUT_CSV}")
print(f"datasets : {df['dataset'].value_counts().to_dict()}")
print(f"scored   : {df['target'].notna().sum()} / {len(df)}")
print(f"\nFeature summary:")
feat_cols = [c for c in df.columns if c not in
             ("filename","dataset","target","fall","n_frames","fps",
              "clip_duration_s","pose_coverage","n_camera_shots",
              "n_cut_frames","dup_frame_frac")]
print(df[feat_cols].describe().T[["mean","std","min","max"]].round(3).to_string())


## Sanity check — signals for one clip

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Change to any stem that exists in POSE_DIR
PLOT_STEM = sorted(POSE_DIR.glob("*_pose.npy"))[0].stem.replace("_pose", "")
print("Plotting:", PLOT_STEM)

pose_seq = np.load(POSE_DIR / f"{PLOT_STEM}_pose.npy")
meta     = json.loads((POSE_DIR / f"{PLOT_STEM}_pose_meta.json").read_text())
fps      = float(meta.get("fps") or 59.94)
cut_set  = set(int(c) for c in (meta.get("cut_frames") or []))
n        = len(pose_seq)
t_axis   = np.arange(n) / fps

# --- shoulder x-diff (for swap visualisation) ---
sh_xdiff = np.array([
    float(frame[L_SH, 0] - frame[R_SH, 0])
    if visible(frame, L_SH, R_SH) else np.nan
    for frame in pose_seq])

# --- shoulder axis angle (unwrapped, cut-masked) ---
angles, t_idx = [], []
for t, frame in enumerate(pose_seq):
    if visible(frame, L_SH, R_SH):
        v = frame[R_SH, :2] - frame[L_SH, :2]
        angles.append(np.degrees(np.arctan2(v[1], v[0])))
        t_idx.append(t)
unwrapped = np.degrees(np.unwrap(np.deg2rad(angles))) if angles else []

# --- normalised wrist-ankle distance ---
heights  = np.array([body_height_px(f) for f in pose_seq])
median_h = float(np.nanmedian(heights))
grab_dists = np.full(n, np.nan)
for t, frame in enumerate(pose_seq):
    cands = []
    for w in [L_WR, R_WR]:
        for a in [L_ANK, R_ANK]:
            if visible(frame, w, a):
                cands.append(np.linalg.norm(frame[w, :2] - frame[a, :2]))
    if cands and not np.isnan(median_h):
        grab_dists[t] = min(cands) / median_h

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
fig.suptitle(f"{PLOT_STEM}   ({n} frames @ {fps:.1f} fps, "
             f"{len(cut_set)} cuts)", fontsize=11)

# cut lines helper
def vcuts(ax):
    for c in cut_set:
        ax.axvline(c / fps, color="red", lw=0.8, ls="--", alpha=0.5)

# spin
if angles:
    axes[0].plot([t / fps for t in t_idx], unwrapped, color="steelblue", lw=1.0)
    axes[0].axhline(0, color="gray", lw=0.5, ls="--")
    for m in [-720, -360, 360, 720]:
        axes[0].axhline(m, color="lightgray", lw=0.4, ls=":")
    total = abs(unwrapped[-1] - unwrapped[0])
    axes[0].set_title(f"Shoulder axis — total {total:.0f}° "
                      f"({total/360:.2f} turns)")
vcuts(axes[0])
axes[0].set_ylabel("Cumul. angle (°)")

# shoulder x-diff + swap threshold
axes[1].plot(t_axis, sh_xdiff, color="darkorange", lw=0.8)
axes[1].axhline(0,  color="gray",  lw=0.8, ls="-")
w = int(meta.get("width") or 1280)
axes[1].axhline( w * 0.10, color="gray", lw=0.5, ls=":")
axes[1].axhline(-w * 0.10, color="gray", lw=0.5, ls=":")
axes[1].set_ylabel("L_sh_x − R_sh_x (px)")
axes[1].set_title("Shoulder x-order  (zero crossing = 180° rotation)")
vcuts(axes[1])

# grab
axes[2].plot(t_axis, grab_dists, color="seagreen", lw=0.8)
axes[2].axhline(GRAB_THRESH, color="red", lw=0.8, ls="--",
                label=f"grab threshold ({GRAB_THRESH})")
axes[2].set_ylabel("Wrist–ankle / height")
axes[2].set_xlabel("Time (s)")
axes[2].set_title("Grab proximity  (below red = grabbing)")
axes[2].legend(fontsize=8)
vcuts(axes[2])

plt.tight_layout()
fig_path = Path(f"feature_sanity_{PLOT_STEM}.png")
plt.savefig(fig_path, dpi=120)
plt.close()
print(f"Saved → {fig_path}")

try:
    from IPython.display import Image, display
    display(Image(filename=str(fig_path)))
except Exception:
    pass


## Quick correlation check against target score

Only for scored clips — purely diagnostic, do not tune on this.

In [ ]:
df = pd.read_csv(OUT_CSV)
scored = df[df["target"].notna()].copy()
print(f"Scored clips: {len(scored)}")
print(f"By dataset: {scored['dataset'].value_counts().to_dict()}")

# Coverage-normalized rotation: swap_turns / pose_coverage compensates for
# missed frames that would have shown more rotation.
scored["swap_turns_cov"] = scored["swap_turns"] / scored["pose_coverage"].clip(lower=0.1)
scored["spin_turns_cov"] = scored["spin_total_turns"] / scored["pose_coverage"].clip(lower=0.1)

feat_cols = [
    "swap_turns", "swap_turns_cov", "swap_rate_hz",
    "spin_total_turns", "spin_turns_cov", "spin_speed_p95_dps",
    "inverted_frac", "cork_detected", "sh_mid_y_std_norm",
    "grab_detected", "grab_duration_frac", "min_grab_dist_norm",
    "amplitude_norm",
    "landing_jitter_norm", "sudden_stop",
    "pose_coverage",
]
feat_cols = [c for c in feat_cols if c in scored.columns]

corr = (scored[feat_cols + ["target"]]
        .corr(method="spearman")["target"]
        .drop("target")
        .sort_values(key=abs, ascending=False))

print("\nSpearman ρ with target score (whole dataset, no CV):")
print(corr.round(3).to_string())
print("\nBy dataset:")
for ds, grp in scored.groupby("dataset"):
    c = (grp[feat_cols + ["target"]]
         .corr(method="spearman")["target"]
         .drop("target"))
    print(f"  {ds} (n={len(grp)}):")
    print("  " + c.round(3).sort_values(key=abs, ascending=False).to_string())
    print()
